## Setup

In [1]:
!pip install neurokit2
import neurokit2 as nk
import pandas as pd
import numpy as np
import os

## Download & Extract Dataset

In [2]:
!wget -q "https://zenodo.org/records/15906524/files/mimic_perform_af_csv.zip?download=1" -O af_csv.zip
!wget -q "https://zenodo.org/records/15906524/files/mimic_perform_non_af_csv.zip?download=1" -O non_af_csv.zip
print("downloaded")

downloaded


In [3]:
!unzip -oq af_csv.zip -d af_data
!unzip -oq non_af_csv.zip -d non_af_data
print("extracted")

extracted


In [4]:
af_files = [f for f in os.listdir('af_data/mimic_perform_af_csv') if f.endswith('_data.csv')]
non_af_files = [f for f in os.listdir('non_af_data/mimic_perform_non_af_csv') if f.endswith('_data.csv')]
print(len(af_files), len(non_af_files))

19 16


## Feature Engineering (PPG only)

In [5]:
def get_pulse_intervals(pleth_segment, sampling_rate=125):
    try:
        pleth_clean = nk.ppg_clean(pleth_segment, sampling_rate=sampling_rate)
        peaks_info, _ = nk.ppg_peaks(pleth_clean, sampling_rate=sampling_rate)
        peak_idx = np.where(peaks_info['PPG_Peaks'] == 1)[0]
        if len(peak_idx) < 8:
            return None
        pp_intervals = np.diff(peak_idx / sampling_rate)
        pp_intervals = pp_intervals[(pp_intervals > 0.3) & (pp_intervals < 1.8)]
        if len(pp_intervals) < 7:
            return None
        diffs = np.diff(pp_intervals)
        if np.max(np.abs(diffs)) > 0.5:
            return None
        return pp_intervals
    except Exception:
        return None

In [6]:
def get_pulse_features_v2(pleth_segment, sampling_rate=125):
    intervals = get_pulse_intervals(pleth_segment, sampling_rate)
    if intervals is None or len(intervals) < 5:
        return None
    diffs = np.diff(intervals)
    mean_int = np.mean(intervals)
    std_int = np.std(intervals)
    rmssd = np.sqrt(np.mean(diffs**2))
    pnn50 = np.mean(np.abs(diffs) > 0.05)
    sd1 = np.std(diffs) / np.sqrt(2)
    sd2_sq = 2 * std_int**2 - sd1**2
    sd2 = np.sqrt(sd2_sq) if sd2_sq > 0 else 0
    hist, _ = np.histogram(intervals, bins=8, density=True)
    hist = hist[hist > 0]
    shannon_entropy = -np.sum(hist * np.log(hist)) if len(hist) > 0 else 0
    return np.array([mean_int, std_int, rmssd, std_int/mean_int, rmssd/mean_int,
                      len(intervals), pnn50, sd1, sd2, shannon_entropy])

## Process Each Patient: Segment into 30-second Windows, Extract Features

In [7]:
def process_patient_file(filepath, label, sampling_rate=125, segment_size=None):
    if segment_size is None:
        segment_size = sampling_rate * 30
    df = pd.read_csv(filepath)
    pleth = df['PPG'].values
    X_list, y_list = [], []
    for start in range(0, len(pleth) - segment_size, segment_size):
        segment = pleth[start:start+segment_size]
        feats = get_pulse_features_v2(segment, sampling_rate=sampling_rate)
        if feats is None:
            continue
        X_list.append(feats)
        y_list.append(label)
    if len(X_list) == 0:
        return np.empty((0, 10)), np.empty((0,))
    return np.array(X_list, dtype=np.float32), np.array(y_list)

In [8]:
X_parts, y_parts = [], []
for f in af_files:
    X_p, y_p = process_patient_file(f'af_data/mimic_perform_af_csv/{f}', label=1)
    if len(X_p) > 0:
        X_parts.append(X_p); y_parts.append(y_p)
for f in non_af_files:
    X_p, y_p = process_patient_file(f'non_af_data/mimic_perform_non_af_csv/{f}', label=0)
    if len(X_p) > 0:
        X_parts.append(X_p); y_parts.append(y_p)

X_all = np.concatenate(X_parts)
y_all = np.concatenate(y_parts)

np.save('X_all_mimicperform.npy', X_all)
np.save('y_all_mimicperform.npy', y_all)

print(X_all.shape)
print(f"AF fraction: {y_all.mean()*100:.1f}%")

/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

(805, 10)
AF fraction: 36.9%


## Train/Test Split (by patient) & Random Forest Training

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_curve

patient_labels = []
patient_ids = []
for f in af_files:
    patient_ids.append(f)
    patient_labels.append(1)
for f in non_af_files:
    patient_ids.append(f)
    patient_labels.append(0)

train_patients, test_patients = train_test_split(
    patient_ids, test_size=0.3, stratify=patient_labels, random_state=42
)
print(len(train_patients), len(test_patients))

24 11


In [10]:
def build_set(patient_files):
    X_parts, y_parts = [], []
    for f in patient_files:
        if f in af_files:
            X_p, y_p = process_patient_file(f'af_data/mimic_perform_af_csv/{f}', label=1)
        else:
            X_p, y_p = process_patient_file(f'non_af_data/mimic_perform_non_af_csv/{f}', label=0)
        if len(X_p) > 0:
            X_parts.append(X_p)
            y_parts.append(y_p)
    return np.concatenate(X_parts), np.concatenate(y_parts)

X_train_mp, y_train_mp = build_set(train_patients)
X_test_mp, y_test_mp = build_set(test_patients)

print(X_train_mp.shape, X_test_mp.shape)
print(f"Train AF: {y_train_mp.mean()*100:.1f}%, Test AF: {y_test_mp.mean()*100:.1f}%")

/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 291 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 25 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 38 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 29 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 10 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib

(543, 10) (262, 10)
Train AF: 31.9%, Test AF: 47.3%


In [11]:
class_weights_mp = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train_mp), y=y_train_mp
)
class_weights_dict_mp = dict(enumerate(class_weights_mp))

rf_mp = RandomForestClassifier(
    n_estimators=300, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1
)
rf_mp.fit(X_train_mp, y_train_mp)

y_pred_mp = rf_mp.predict(X_test_mp)
print(classification_report(y_test_mp, y_pred_mp, target_names=['Non-AF', 'AF']))

              precision    recall  f1-score   support

      Non-AF       0.91      1.00      0.95       138
          AF       1.00      0.89      0.94       124

    accuracy                           0.95       262
   macro avg       0.95      0.94      0.95       262
weighted avg       0.95      0.95      0.95       262



## Threshold Tuning

In [12]:
y_pred_probs_mp = rf_mp.predict_proba(X_test_mp)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test_mp, y_pred_probs_mp)
f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1s)
print(f"Best threshold: {thresholds[best_idx]:.3f}, Precision: {precisions[best_idx]:.3f}, Recall: {recalls[best_idx]:.3f}, F1: {f1s[best_idx]:.3f}")

Best threshold: 0.233, Precision: 0.983, Recall: 0.944, F1: 0.963


## Sanity Checks (leakage check, feature separation)

In [13]:
train_set = set(train_patients)
test_set = set(test_patients)
print("Overlap:", train_set & test_set)

import pandas as pd
feat_names = ['mean_int', 'std_int', 'rmssd', 'cv', 'rmssd_ratio', 'n_beats', 'pnn50', 'sd1', 'sd2', 'entropy']
df_check = pd.DataFrame(X_train_mp, columns=feat_names)
df_check['label'] = y_train_mp
print(df_check.groupby('label')['rmssd_ratio'].describe())

Overlap: set()
       count      mean       std       min       25%       50%       75%  \
label                                                                      
0      370.0  0.054440  0.044533  0.010690  0.026340  0.041035  0.063009   
1      173.0  0.282537  0.055679  0.086407  0.240954  0.288008  0.322600   

            max  
label            
0      0.290715  
1      0.387760  


## 5-Fold Cross-Validation (PPG only)

In [14]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(patient_ids, patient_labels)):
    fold_train = [patient_ids[i] for i in train_idx]
    fold_test = [patient_ids[i] for i in test_idx]
    X_tr, y_tr = build_set(fold_train)
    X_te, y_te = build_set(fold_test)
    rf_fold = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
    rf_fold.fit(X_tr, y_tr)
    y_pred_fold = rf_fold.predict(X_te)
    f1 = classification_report(y_te, y_pred_fold, output_dict=True)['1']['f1-score']
    scores.append(f1)
    print(f"Fold {fold+1}: F1 = {f1:.3f}")

print(f"Mean F1 across folds: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 1: F1 = 1.000


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 2: F1 = 0.953


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 3: F1 = 0.975


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 4: F1 = 0.983


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 31 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 21 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 10 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 6 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 13 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/p

Fold 5: F1 = 0.977
Mean F1 across folds: 0.978 ± 0.015


In [15]:
scores_f1 = []
scores_acc = []

for fold, (train_idx, test_idx) in enumerate(skf.split(patient_ids, patient_labels)):
    fold_train = [patient_ids[i] for i in train_idx]
    fold_test = [patient_ids[i] for i in test_idx]
    X_tr, y_tr = build_set(fold_train)
    X_te, y_te = build_set(fold_test)
    rf_fold = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
    rf_fold.fit(X_tr, y_tr)
    y_pred_fold = rf_fold.predict(X_te)
    report = classification_report(y_te, y_pred_fold, output_dict=True)
    f1 = report['1']['f1-score']
    acc = report['accuracy']
    scores_f1.append(f1)
    scores_acc.append(acc)
    print(f"Fold {fold+1}: F1 = {f1:.3f}, Accuracy = {acc:.3f}")

print(f"\nMean F1: {np.mean(scores_f1):.3f} ± {np.std(scores_f1):.3f}")
print(f"Mean Accuracy: {np.mean(scores_acc):.3f} ± {np.std(scores_acc):.3f}")

/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 1: F1 = 1.000, Accuracy = 1.000


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 2: F1 = 0.953, Accuracy = 0.969


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 3: F1 = 0.975, Accuracy = 0.983


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 67 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 16 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 96 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 19 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 33 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/

Fold 4: F1 = 0.983, Accuracy = 0.980


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 31 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 21 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 10 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 6 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 13 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(
/usr/local/lib/p

Fold 5: F1 = 0.977, Accuracy = 0.989

Mean F1: 0.978 ± 0.015
Mean Accuracy: 0.984 ± 0.010


## Extension: PPG + Respiration

In [16]:
def get_resp_features(resp_segment, sampling_rate=125):
    try:
        resp_clean = nk.rsp_clean(resp_segment, sampling_rate=sampling_rate)
        _, resp_info = nk.rsp_peaks(resp_clean, sampling_rate=sampling_rate)
        peak_idx = resp_info['RSP_Peaks']

        if len(peak_idx) < 3:
            return None

        breath_intervals = np.diff(peak_idx / sampling_rate)
        breath_intervals = breath_intervals[(breath_intervals > 1.0) & (breath_intervals < 15.0)]

        if len(breath_intervals) < 2:
            return None

        mean_breath = np.mean(breath_intervals)
        std_breath = np.std(breath_intervals)
        resp_rate = 60 / mean_breath  

        return np.array([mean_breath, std_breath, std_breath/mean_breath, resp_rate])
    except Exception:
        return None

In [20]:
def process_patient_file_multi(filepath, label, sampling_rate=125, segment_size=None):
    if segment_size is None:
        segment_size = sampling_rate * 30

    df = pd.read_csv(filepath)
    if 'resp' not in df.columns:
        return np.empty((0, 14)), np.empty((0,))

    pleth = df['PPG'].values
    resp = df['resp'].values

    X_list, y_list = [], []
    for start in range(0, len(pleth) - segment_size, segment_size):
        ppg_feats = get_pulse_features_v2(pleth[start:start+segment_size], sampling_rate=sampling_rate)
        if ppg_feats is None:
            continue
        resp_feats = get_resp_features(resp[start:start+segment_size], sampling_rate=sampling_rate)
        if resp_feats is None:
            continue
        combined = np.concatenate([ppg_feats, resp_feats])
        X_list.append(combined)
        y_list.append(label)

    if len(X_list) == 0:
        return np.empty((0, 14)), np.empty((0,))
    return np.array(X_list, dtype=np.float32), np.array(y_list)

In [19]:
missing_resp = []
for f in af_files + non_af_files:
    folder = 'af_data/mimic_perform_af_csv' if f in af_files else 'non_af_data/mimic_perform_non_af_csv'
    cols = pd.read_csv(os.path.join(folder, f), nrows=1).columns.tolist()
    if 'resp' not in cols:
        missing_resp.append((f, cols))

print(len(missing_resp))
for f, cols in missing_resp[:5]:
    print(f, cols)

9
mimic_perform_af_017_data.csv ['Time', 'PPG', 'ECG']
mimic_perform_af_007_data.csv ['Time', 'PPG', 'ECG']
mimic_perform_af_002_data.csv ['Time', 'PPG', 'ECG']
mimic_perform_af_006_data.csv ['Time', 'PPG', 'ECG']
mimic_perform_af_005_data.csv ['Time', 'PPG', 'ECG']


In [ ]:
def build_set_multi(patient_files):
    X_parts, y_parts = [], []
    for f in patient_files:
        if f in af_files:
            X_p, y_p = process_patient_file_multi(f'af_data/mimic_perform_af_csv/{f}', label=1)
        else:
            X_p, y_p = process_patient_file_multi(f'non_af_data/mimic_perform_non_af_csv/{f}', label=0)
        if len(X_p) > 0:
            X_parts.append(X_p)
            y_parts.append(y_p)
    return np.concatenate(X_parts), np.concatenate(y_parts)


In [21]:
scores_f1_multi = []
scores_acc_multi = []

for fold, (train_idx, test_idx) in enumerate(skf.split(patient_ids, patient_labels)):
    fold_train = [patient_ids[i] for i in train_idx]
    fold_test = [patient_ids[i] for i in test_idx]
    X_tr, y_tr = build_set_multi(fold_train)
    X_te, y_te = build_set_multi(fold_test)
    rf_fold = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
    rf_fold.fit(X_tr, y_tr)
    y_pred_fold = rf_fold.predict(X_te)
    report = classification_report(y_te, y_pred_fold, output_dict=True)
    scores_f1_multi.append(report['1']['f1-score'])
    scores_acc_multi.append(report['accuracy'])
    print(f"Fold {fold+1}: F1 = {report['1']['f1-score']:.3f}, Accuracy = {report['accuracy']:.3f}")

print(f"\nMean F1 (PPG+Resp): {np.mean(scores_f1_multi):.3f} ± {np.std(scores_f1_multi):.3f}")
print(f"Mean Accuracy (PPG+Resp): {np.mean(scores_acc_multi):.3f} ± {np.std(scores_acc_multi):.3f}")

/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 291 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(


Fold 1: F1 = 1.000, Accuracy = 1.000


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 291 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(


Fold 2: F1 = 0.918, Accuracy = 0.964


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 291 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(


Fold 3: F1 = 0.974, Accuracy = 0.983


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 291 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(


Fold 4: F1 = 0.979, Accuracy = 0.978


/usr/local/lib/python3.12/dist-packages/neurokit2/ppg/ppg_clean.py:100: NeuroKitWarning: There are 291 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(


Fold 5: F1 = 0.950, Accuracy = 0.975

Mean F1 (PPG+Resp): 0.964 ± 0.028
Mean Accuracy (PPG+Resp): 0.980 ± 0.012
